# 28_02 고장 예측 결과 해석

In [1]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


In [3]:
# 28_1장 기본 머신러닝 코드 재현
df = pd.read_csv('28_cmapss_fd001_sample.csv')
feature_cols = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_15']
X = df[feature_cols]
y = df["failure_soon"]

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

Xtr, X_test, ytr, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(random_state=42).fit(Xtr,ytr)
y_pred = model.predict(X_test)

In [ ]:
# from sklearn.metrics import confusion_matrix

# cm = confusion_matrix(y_test, y_pred)

# tn, fp, fn, tp = cm.ravel()

# 01 F1-score와 분류 리포트
F1-score란 무엇인가 · 분류 리포트 읽는 순서


### F1 출력
고장 임박 클래스의 F1을 함수 한 줄로 계산해 출력


In [19]:
# 코드
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred)
print("고장 임박 F1:", round(f1, 4))

고장 임박 F1: 0.6486


### 리포트 출력과 해석
클래스별 지표를 표로 출력하고 한 줄씩 해석 적기


In [21]:
# 코드
from sklearn.metrics import classification_report
report = classification_report(y_test, y_pred, target_names=['정상', '고장임박'])
print(report)


              precision    recall  f1-score   support

          정상       0.90      0.94      0.92       240
        고장임박       0.71      0.60      0.65        60

    accuracy                           0.87       300
   macro avg       0.80      0.77      0.78       300
weighted avg       0.86      0.87      0.87       300



# 02 임계값 조정
임계값을 조정한다는 것 · 임계값과 정밀도·재현율


### 확률 추출과 임계값 적용
고장(1) 확률을 꺼내 임계값별 예측 생성


In [30]:
# 코드
proba = model.predict_proba(X_test)[:, 1]

for t in [0.3, 0.5, 0.7]:
  pred_t = (proba >= t).astype(int)
  # 확률이 t보다 큰 것이 몇 개인가?
  print(f"전체 데이터 중 {t}이상인 경우는 {pred_t.sum()}개")

전체 데이터 중 0.3이상인 경우는 86개
전체 데이터 중 0.5이상인 경우는 55개
전체 데이터 중 0.7이상인 경우는 27개


### 임계값별 지표 비교
각 임계값의 재현율·정밀도·F1 출력


In [31]:
# 코드
from sklearn.metrics import precision_score, recall_score, f1_score
for t in [0.3, 0.5, 0.7]:
  pt = (proba >= t).astype(int)
  print(t, round(recall_score(y_test, pt), 3),
  round(precision_score(y_test, pt), 3))

0.3 0.817 0.57
0.5 0.65 0.709
0.7 0.383 0.852


### 비교표 생성
임계값별 지표와 FP·FN을 한 표로


In [33]:
# 코드
from sklearn.metrics import confusion_matrix, recall_score, precision_score

rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.7]:
  pt = (proba >= t).astype(int)
  tn, fp, fn, tp = confusion_matrix(y_test, pt).ravel()
  rows.append([t, recall_score(y_test, pt), fp, fn])
print(pd.DataFrame(rows, columns=["임계값","재현율","FP","FN"]))


   임계값       재현율  FP  FN
0  0.2  0.900000  63   6
1  0.3  0.816667  37  11
2  0.4  0.733333  26  16
3  0.5  0.650000  16  21
4  0.7  0.383333   4  37


# 03 이상탐지 평가와 결과 해석
이상탐지 결과 평가 · 정비 의사결정 해석


### IsolationForest 실행
MIMII 특징으로 이상탐지 학습·예측


In [ ]:
# 코드
from sklearn.ensemble import IsolationForest
dfm = pd.read_csv("28_mimii_features_sample.csv")
# dfm.head() 여기서 컬럼을 확인해서 X값 판단
Xm = dfm[["rms", "spectral_centroid", "zero_crossing_rate"]]
raw = IsolationForest(contamination=0.1, random_state=42).fit_predict(Xm)

### 변환과 평가
-1을 1(이상)로 바꿔 라벨과 비교


In [36]:
# 코드
y_true = dfm["label"]
y_iso = (raw == -1).astype(int) # -1(이상)->1, 1(정상)->0

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_true, y_iso))
print(classification_report(y_true, y_iso, target_names=["정상","이상"]))

[[591  36]
 [129  44]]
              precision    recall  f1-score   support

          정상       0.82      0.94      0.88       627
          이상       0.55      0.25      0.35       173

    accuracy                           0.79       800
   macro avg       0.69      0.60      0.61       800
weighted avg       0.76      0.79      0.76       800



### MIMII 로드와 분할
MIMII 특징과 라벨로 학습 준비 — 지도학습 모드


In [ ]:
# 코드
from sklearn.model_selection import train_test_split
dfm = pd.read_csv("28_mimii_features_sample.csv")
Xm = dfm[["rms", "spectral_centroid", "zero_crossing_rate"]]
ym = dfm["label"]
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3, random_state=42, stratify=ym)

629    0
33     0
118    0
481    1
609    0
Name: label, dtype: int64

### 학습과 평가
RandomForest 학습 후 리포트 출력 — 동일 절차


In [46]:
# 코드
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf = RandomForestClassifier(random_state=42).fit(Xtr, ytr)

pred = clf.predict(Xte)
# pred[:15]
pred = clf.predict(Xte)

print(classification_report(yte, pred, target_names=["정상","이상"]))

              precision    recall  f1-score   support

          정상       0.97      0.99      0.98       188
          이상       0.98      0.88      0.93        52

    accuracy                           0.97       240
   macro avg       0.97      0.94      0.96       240
weighted avg       0.97      0.97      0.97       240



### 평가 함수 정의
모델과 평가 데이터를 받는 `evaluate()` 함수


In [51]:
# 코드
from sklearn.metrics import recall_score, precision_score, confusion_matrix

def evaluate(model, X_test, y_test, name="모델"):
  pred = model.predict(X_test)
  tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
  
  print(name, "재현율", round(recall_score(y_test, pred), 3), "FN", fn)

### 함수로 한 번에 평가
함수에 모델을 넣어 평가 실행 — 모델만 바꿔도 작동


In [53]:
# 코드
evaluate(model, X_test, y_test, "RandomForest")

from sklearn.metrics import classification_report

print(classification_report(y_test, model.predict(X_test),
target_names=["정상","고장임박"]))

RandomForest 재현율 0.6 FN 24
              precision    recall  f1-score   support

          정상       0.90      0.94      0.92       240
        고장임박       0.71      0.60      0.65        60

    accuracy                           0.87       300
   macro avg       0.80      0.77      0.78       300
weighted avg       0.86      0.87      0.87       300



### 두 임계값 예측 생성
임계값 0.5와 0.3으로 예측을 만들고 재현율 비교


In [56]:
# 코드
proba = model.predict_proba(X_test)[:, 1]
pred_05 = (proba >= 0.5).astype(int)
pred_03 = (proba >= 0.3).astype(int)
print("0.5", pred_05.sum())
print("0.3", pred_03.sum())

from sklearn.metrics import recall_score, precision_score
print("0.5", round(recall_score(y_test, pred_05), 3))
print("0.3", round(recall_score(y_test, pred_03), 3))

0.5 55
0.3 86
0.5 0.65
0.3 0.817


### 리포트 초안 프롬프트
평가 수치를 정비 리포트 초안으로 정리하는 AI 프롬프트


In [18]:
# 코드